In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install transformers

In [3]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
import shutil
import string
from sklearn.metrics import classification_report
import tensorflow as tf
from keras.models import Model
from tensorflow.keras import layers
from tensorflow.keras import losses
from tensorflow.keras.optimizers import Adam
from keras.callbacks import EarlyStopping,ModelCheckpoint
from tensorflow.keras.layers import Dense, Input, Dropout, Bidirectional, LSTM, Embedding, BatchNormalization,  Reshape, Conv2D, MaxPool2D, concatenate, Flatten, Activation
import torch
import numpy as np
from transformers import BertTokenizer, BertModel,RobertaTokenizer, RobertaModel,AutoTokenizer, AutoModel
import ast

In [4]:
# Verificar si la GPU está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
archivo_3 = '/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/Tuning/BestModels/finallDataset.csv'
train_full = pd.read_csv(archivo_3)

In [6]:
train_full

,Problema,Solución,y
0,Write a Python function that returns the facto...,def factorial(n):\n result = 0\n i = 1\n...,0
1,Write a Python function that returns the sum o...,def sum_1_to_100():\n total = 0\n i = 1\...,0
2,Write a Python function that prints the number...,def print_and_store():\n numbers = []\n ...,0
3,Write a Python function to print the numbers f...,def print_and_store_reverse():\n numbers = ...,0
4,Write a Python function to check if a number i...,def is_prime(num):\n i = 2\n while i < n...,0
...,...,...,...
4130,calculate how many numbers are divisible by 3,def buggy_count_mult_3(n):\n i = 0\n cou...,6
4131,calculate how many numbers are divisible by 3,def buggy_count_mult_3(n):\n i = 0\n cou...,6
4132,calculate how many numbers are divisible by 3,def buggy_count_mult_3(n):\n count = 0\n ...,6
4133,calculate how many numbers are divisible by 3,def buggy_count_mult_3(n):\n count = 0\n ...,6


# AST spliter D. Gries form

In [7]:
class WhileLoopFinder(ast.NodeVisitor):
    def __init__(self, source_code):
        self.source_code = source_code.splitlines()
        self.functions_with_while = []
        self.current_function = None

    def visit_FunctionDef(self, node):
        original_current_function = self.current_function
        self.current_function = {
            "function_name": node.name,
            "has_while_loop": False,
            "pre_while_code_lines": [],
            "while_loops": []
        }

        function_start_line = node.lineno - 1
        function_end_line = (node.end_lineno if hasattr(node, 'end_lineno') else len(self.source_code))
        function_lines = self.source_code[function_start_line:function_end_line]

        found_while = False
        for stmt in node.body:
            if isinstance(stmt, ast.While):
                self.current_function["has_while_loop"] = True
                found_while = True
                try:
                    while_condition = ast.unparse(stmt.test).strip()
                    print("OK condition")
                except AttributeError:
                    print("Error al obtener la condición del while")
                try:
                    while_body_code = ast.unparse(ast.Module(body=stmt.body, type_ignores=[])).strip()
                    print("Ok body")
                except AttributeError:
                    print("Error al obtener el código del cuerpo del while")

                self.current_function["while_loops"].append({
                    "condition": while_condition,
                    "body_code": while_body_code
                })
            elif not found_while:
                start_line = stmt.lineno - 1
                end_line = (stmt.end_lineno if hasattr(stmt, 'end_lineno') else stmt.lineno)
                self.current_function["pre_while_code_lines"].extend(self.source_code[start_line:end_line])
            self.generic_visit(stmt)

        if self.current_function["has_while_loop"]:
            last_while_end_line = None
            if self.current_function["while_loops"]:
                last_while_node = next((node for node in reversed(node.body) if isinstance(node, ast.While)), None)
                if last_while_node and hasattr(last_while_node, 'end_lineno'):
                    last_while_end_line = last_while_node.end_lineno
            post_while_code_lines = []
            if last_while_end_line:
                function_indent = len(self.source_code[node.lineno - 1]) - len(self.source_code[node.lineno - 1].lstrip())
                for line in self.source_code[last_while_end_line:function_end_line]:
                    if line.startswith(self.source_code[node.lineno - 1][:function_indent] + "    "):
                        post_while_code_lines.append(line[function_indent + 4:])
                    else:
                        post_while_code_lines.append(line[function_indent:])
            self.current_function["post_while_code_lines"] = "\n".join(post_while_code_lines).strip()
            self.current_function["pre_while_code_lines"] = "\n".join(self.current_function["pre_while_code_lines"]).strip()
            self.functions_with_while.append(self.current_function)

        self.current_function = original_current_function



In [8]:
def DGries_states(solution):
  try:
    # Crear el AST
    tree = ast.parse(solution)
    # Recorrer el AST con nuestro visitante
    finder = WhileLoopFinder(solution)
    finder.visit(tree)
    for func_info in finder.functions_with_while:
      initial_state=func_info["pre_while_code_lines"] if func_info["pre_while_code_lines"] else False
      end_state=""
      transformation_state=""
      for j, wl in enumerate(func_info["while_loops"]):
          end_state=wl["condition"]
          transformation_state=wl["body_code"]

      if not initial_state:
        initial_state=transformation_state
      return initial_state,transformation_state,end_state
  except Exception as error:
    return "Exception:"+str(error),"Exception:"+str(error),"Exception:"+str(error)




# Encoder Description and Code

In [9]:
class Encoder:
  def __init__(self):
    self.is_loadtokenizers=False



  def tokenize_and_generate_embeddings_descriptions(self,description):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input description and move tokens to GPU
    tokens = self.beart_tokenizer.encode_plus(description, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.beart_model(**tokens)
    # Extract embeddings for all tokens
    desc_embeddings = outputs.last_hidden_state.cpu().numpy()
    return desc_embeddings

  def tokenize_and_generate_embeddings_codes(self,code):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input code and move tokens to GPU
    tokens = self.codebeart_tokenizer.encode_plus(code, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.codebeart_model(**tokens)
    # Extract embeddings for all tokens
    code_embeddings = outputs.last_hidden_state.cpu().numpy()

    return code_embeddings


  def tokenize_and_generate_embeddings_graphcodes(self,code):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input code and move tokens to GPU
    tokens = self.graphcodebert_tokenizer.encode_plus(code, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.graphcodebert_model(**tokens)
    # Extract embeddings for all tokens
    code_embeddings = outputs.last_hidden_state.cpu().numpy()
    return code_embeddings

  def load_beart_tokenizer(self):
    # Verificar si la GPU está disponible
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Cargar el tokenizer y el modelo en la GPU si está disponible
    model_name = "bert-base-uncased"
    self.beart_model = BertModel.from_pretrained(model_name)
    self.beart_tokenizer = BertTokenizer.from_pretrained(model_name)
    self.beart_model.to(device)

  def load_graphcodebert_tokenizer(self):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    self.graphcodebert_tokenizer = AutoTokenizer.from_pretrained("microsoft/graphcodebert-base")
    self.graphcodebert_model = AutoModel.from_pretrained("microsoft/graphcodebert-base")
    self.graphcodebert_model.to(device)


  def load_codebert_tokenizer(self):
    # Verificar si la GPU está disponible
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Cargar el tokenizer y el modelo en la GPU si está disponible
    self.codebeart_tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")
    self.codebeart_model = RobertaModel.from_pretrained("microsoft/codebert-base")
    self.codebeart_model.to(device)

  def start_tokenizer(self):
    self.load_beart_tokenizer()
    #self.load_codebert_tokenizer()
    self.load_graphcodebert_tokenizer()
    self.is_loadtokenizers=True




# All characteristics

In [10]:
train_full.head()

,Problema,Solución,y
0,Write a Python function that returns the facto...,def factorial(n):\n result = 0\n i = 1\n...,0
1,Write a Python function that returns the sum o...,def sum_1_to_100():\n total = 0\n i = 1\...,0
2,Write a Python function that prints the number...,def print_and_store():\n numbers = []\n ...,0
3,Write a Python function to print the numbers f...,def print_and_store_reverse():\n numbers = ...,0
4,Write a Python function to check if a number i...,def is_prime(num):\n i = 2\n while i < n...,0


In [11]:
train_full[["Estado incial","Transformación de estado","Estado final"]]=train_full['Solución'].apply(DGries_states).apply(pd.Series)

Streaming output truncated to the last 5000 lines.
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK c

In [12]:
train_full.dropna(inplace=True)

In [13]:
encoder=Encoder()
encoder.start_tokenizer()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
%%time
problem=train_full['Problema'].apply(encoder.tokenize_and_generate_embeddings_descriptions).to_numpy()
code=train_full['Solución'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
startstate = train_full['Estado incial'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
finalstate = train_full['Estado final'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
transstate = train_full['Transformación de estado'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()

CPU times: user 3min 8s, sys: 1.23 s, total: 3min 9s
Wall time: 3min 37s


In [15]:
problem.shape,code.shape,startstate.shape,finalstate.shape,transstate.shape

((4094,), (4094,), (4094,), (4094,), (4094,))

In [16]:
print(startstate.shape)
print(startstate[1262].shape)


(4094,)
(1, 13, 768)


In [17]:
Xp=np.array([sentence[0].mean(axis=0) for sentence in problem])
Xcode=np.array([sentence[0].mean(axis=0) for sentence in code])
Xs=np.array([sentence[0].mean(axis=0) for sentence in startstate])
Xf=np.array([sentence[0].mean(axis=0) for sentence in finalstate])
Xt=np.array([sentence[0].mean(axis=0) for sentence in transstate])
Xp.shape,Xs.shape,Xf.shape,Xt.shape

((4094, 768), (4094, 768), (4094, 768), (4094, 768))

In [18]:
y=train_full['y']
y=y.to_numpy()
np.unique(y)

array([0, 1, 2, 3, 4, 5, 6])

In [19]:
from sklearn.model_selection import train_test_split

Xp_train,Xp_test,Xcode_train,Xcode_test,Xs_train,Xs_test,Xt_train,Xt_test,Xf_train,Xf_test,y_train,y_test=train_test_split(Xp,Xcode,Xs,Xt,Xf,y,test_size=0.2,random_state=2023, stratify=y)

In [20]:
Xp_train.shape,Xp_test.shape,Xcode_train.shape, Xcode_test.shape, Xs_train.shape,Xs_test.shape,Xt_train.shape,Xt_test.shape,Xf_train.shape,Xf_test.shape,y_train.shape,y_test.shape

((3275, 768),
 (819, 768),
 (3275, 768),
 (819, 768),
 (3275, 768),
 (819, 768),
 (3275, 768),
 (819, 768),
 (3275, 768),
 (819, 768),
 (3275,),
 (819,))

# Keras model

In [21]:
from tensorflow.keras import backend as K
import gc
def ANNTC(input,base,pow_initial,num_max_blocks):

  drop_out=0.5
  print("base",base,"pow",pow_initial,"num_max_blocks",num_max_blocks)
  n=num_max_blocks//2
  neurons=int(base**(pow_initial+n+1))
  # Encoder
  x=None
  print("Encoder",n)
  try:
    for i in range(n):
      x=Dense(neurons)(input)
      x=BatchNormalization()(x)
      x=Activation('relu')(x)
      x=Dropout(drop_out)(x)
      input=x
      drop_out=0.2
      print("Block ",i,neurons)
      neurons=int(neurons/base)



    #BottleNeck
    x=Dense(neurons)(x)
    x=BatchNormalization()(x)
    x=Activation('relu')(x)
    x=Dropout(0.2)(x)
    print("BottleNeck",neurons)

    # Decoder
    print("Decoder",n)
    for i in range(n):
      neurons=int(neurons*base)
      print("Block ",i,neurons)
      x=Dense(neurons)(x)
      x=BatchNormalization()(x)
      x=Activation('relu')(x)
      x=Dropout(drop_out)(x)
      if i==n-1:
        drop_out=0.5
      drop_out=0.2
  except Exception as err:

    K.clear_session()
    gc.collect()
    del x
    print(f"Unexpected {err=}, {type(err)=}")
    raise

  return x

def neural_network_all(problem,code,start_input,trass_input,final_input,base=10,pow_initial=4,num_max_blocks=3):
  mlp_p=ANNTC(problem,base,pow_initial,num_max_blocks)
  mlp_code=ANNTC(code,base,pow_initial,num_max_blocks)
  mlp_s=ANNTC(start_input,base,pow_initial,num_max_blocks)
  mlp_t=ANNTC(trass_input,base,pow_initial,num_max_blocks)
  mlp_f=ANNTC(final_input,base,pow_initial,num_max_blocks)
  ## Concatenate All models
  combined = concatenate([mlp_p,mlp_code ,mlp_s, mlp_t,mlp_f])
  ## output layer
  output=Dense(8)(combined)
  output=BatchNormalization()(output)
  output=Activation('softmax')(output)
  ## Model
  model = Model(inputs=[problem_input,code,start_input, trass_input,final_input], outputs=output)
  return model


def neural_network_code_gries(code_input,start_input,trass_input,final_input,base=10,pow_initial=4,num_max_blocks=3):
  mlp_code=ANNTC(code_input,base,pow_initial,num_max_blocks)
  mlp_s=ANNTC(start_input,base,pow_initial,num_max_blocks)
  mlp_t=ANNTC(trass_input,base,pow_initial,num_max_blocks)
  mlp_f=ANNTC(final_input,base,pow_initial,num_max_blocks)
  ## Concatenate All models
  combined = concatenate([mlp_code, mlp_s, mlp_t,mlp_f])
  ## output layer
  output=Dense(8)(combined)
  output=BatchNormalization()(output)
  output=Activation('softmax')(output)
  ## Model
  model = Model(inputs=[code_input, start_input, trass_input,final_input], outputs=output)
  return model

def neural_network(problem,start_input,trass_input,final_input,base=10,pow_initial=4,num_max_blocks=3):
  mlp_p=ANNTC(problem,base,pow_initial,num_max_blocks)
  mlp_s=ANNTC(start_input,base,pow_initial,num_max_blocks)
  mlp_t=ANNTC(trass_input,base,pow_initial,num_max_blocks)
  mlp_f=ANNTC(final_input,base,pow_initial,num_max_blocks)
  ## Concatenate All models
  combined = concatenate([mlp_p, mlp_s, mlp_t,mlp_f])
  ## output layer
  output=Dense(8)(combined)
  output=BatchNormalization()(output)
  output=Activation('softmax')(output)
  ## Model
  model = Model(inputs=[problem_input, start_input, trass_input,final_input], outputs=output)
  return model


def neural_network_wproblem(start_input,trass_input,final_input,base=10,pow_initial=4,num_max_blocks=3):
  mlp_s=ANNTC(start_input,base,pow_initial,num_max_blocks)
  mlp_t=ANNTC(trass_input,base,pow_initial,num_max_blocks)
  mlp_f=ANNTC(final_input,base,pow_initial,num_max_blocks)
  ## Concatenate All models
  combined = concatenate([mlp_s, mlp_t,mlp_f])
  ## output layer
  output=Dense(8)(combined)
  output=BatchNormalization()(output)
  output=Activation('softmax')(output)
  ## Model
  model = Model(inputs=[start_input, trass_input,final_input], outputs=output)
  return model

def neural_network_problem_code(problem,code_input,base=10,pow_initial=4,num_max_blocks=3):
  mlp_p=ANNTC(problem,base,pow_initial,num_max_blocks)
  mlp_code=ANNTC(code_input,base,pow_initial,num_max_blocks)
  ## Concatenate All models
  combined = concatenate([mlp_p, mlp_code])
  ## output layer
  output=Dense(8)(combined)
  output=BatchNormalization()(output)
  output=Activation('softmax')(output)
  ## Model
  model = Model(inputs=[problem, code_input], outputs=output)
  return model

def neural_network_code(code_input,base=10,pow_initial=4,num_max_blocks=3):

  mlp_code=ANNTC(code_input,base,pow_initial,num_max_blocks)
  ## output layer
  output=Dense(8)(mlp_code)
  output=BatchNormalization()(output)
  output=Activation('softmax')(output)
  ## Model
  model = Model(inputs=[code_input], outputs=output)
  return model

In [22]:
import numpy as np
np.linspace(5,70,14)


array([ 5., 10., 15., 20., 25., 30., 35., 40., 45., 50., 55., 60., 65.,
       70.])

In [23]:
models=[]
models_names=["problemall","problem_gries","code_gries","gries","problem_code","code"]

problem_input=Input((768,))
code_input=Input((768,))
start_input=Input((768,))
trass_input=Input((768,))
final_input=Input((768,))

print("problem all")
models.append(neural_network_all(problem_input,code_input,start_input,trass_input,final_input,base=8,pow_initial=1, num_max_blocks=3))
print("\nproblem gries")
models.append(neural_network(problem_input,start_input,trass_input,final_input,base=8,pow_initial=1, num_max_blocks=3))
print("\ncode gries")
models.append(neural_network_code_gries(code_input,start_input,trass_input,final_input,base=8,pow_initial=1, num_max_blocks=3))
print("\ngries")
models.append(neural_network_wproblem(start_input,trass_input,final_input,base=8,pow_initial=1, num_max_blocks=3))
print("\nproblem code")
models.append(neural_network_problem_code(problem_input,code_input,base=8,pow_initial=1, num_max_blocks=3))
print("\ncode")
models.append(neural_network_code(code_input,base=8,pow_initial=1, num_max_blocks=3))


problem all
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512

problem gries
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512

code gries
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNe

In [24]:
from tensorflow.keras.callbacks import ReduceLROnPlateau
from time import time
#tensorflow random state
tf.random.set_seed(2023)

times=[]
max_accuracies=[]
max_val_accuracies=[]
max_val_losses=[]
max_losses=[]
learning_rates=[]
Model_Xtrain=[Xp_train,Xcode_train,Xs_train,Xt_train,Xf_train]

for i,my_model in enumerate(models):
  print(models_names[i])

  problem_input=Input((768,))
  start_input=Input((768,))
  trass_input=Input((768,))
  final_input=Input((768,))

  patience=35

  my_model.compile(loss='sparse_categorical_crossentropy',
                optimizer=Adam(),
                metrics=['accuracy'])

  es = EarlyStopping(monitor='val_loss', mode='min', patience=patience)

  mc = ModelCheckpoint('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/Tuning/BestModels/ablationsGRAPH_finall_{0}.keras'.format(models_names[i]), monitor='val_loss', mode='min', save_best_only=True)
  reduce_lr = ReduceLROnPlateau(
      monitor='val_loss',
      factor=0.1,
      patience=patience//2,
      min_lr=1e-9,
      verbose=1
  )
  if(i==1):
    Model_Xtrain=[Xp_train,Xs_train,Xt_train,Xf_train]
  if(i==2):
    Model_Xtrain=[Xcode_train,Xs_train,Xt_train,Xf_train]
  if(i==3):
    Model_Xtrain=[Xs_train,Xt_train,Xf_train]
  if(i==4):
    Model_Xtrain=[Xp_train,Xcode_train]
  if(i==5):
    Model_Xtrain=[Xcode_train]


  time1=time()
  history=my_model.fit(Model_Xtrain,y_train,batch_size=100,epochs=1000,
            validation_split=0.2,callbacks=[es, mc,reduce_lr])
  time2=time()

  df_histories=pd.DataFrame(history.history)
  df_histories.to_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/Tuning/BestModels/ablations_best_finallGRAPH_{0}.csv'.format(models_names[i]))

  times.append(time2-time1)

  max_accuracies.append(max(history.history['accuracy']))
  max_val_accuracies.append(max(history.history['val_accuracy']))
  max_val_losses.append(max(history.history['val_loss']))
  max_losses.append(max(history.history['loss']))
  learning_rates.append(min(history.history['learning_rate']))

  print(f"{patience}")
  print("times=",time2-time1)
  print("max_val_accuracy",max(history.history['val_accuracy']))
  print("max_accuracy",max(history.history['accuracy']))


  print("max_val_loss",max(history.history['val_loss']))
  print("max_loss",max(history.history['loss']))
  print("learning_rate",min(history.history['learning_rate']))





problemall
Epoch 1/1000
27/27 ━━━━━━━━━━━━━━━━━━━━ 38s 591ms/step - accuracy: 0.2615 - loss: 2.0943 - val_accuracy: 0.4977 - val_loss: 1.8704 - learning_rate: 0.0010
Epoch 2/1000
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.5629 - loss: 1.4802 - val_accuracy: 0.4901 - val_loss: 1.7700 - learning_rate: 0.0010
Epoch 3/1000
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.6769 - loss: 1.2628 - val_accuracy: 0.4687 - val_loss: 1.7168 - learning_rate: 0.0010
Epoch 4/1000
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.7375 - loss: 1.1402 - val_accuracy: 0.5053 - val_loss: 1.5995 - learning_rate: 0.0010
Epoch 5/1000
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.7583 - loss: 1.0635 - val_accuracy: 0.5924 - val_loss: 1.4602 - learning_rate: 0.0010
Epoch 6/1000
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.7945 - loss: 0.9858 - val_accuracy: 0.6534 - val_loss: 1.3363 - learning_rate: 0.0010
Epoch 7/1000
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.82

In [25]:
df_histories=pd.DataFrame({"times":times,"max_accuracies":max_accuracies,"max_val_accuracies":max_val_accuracies,"max_val_losses":max_val_losses,"max_losses":max_losses,"learning_rates":learning_rates})
df_histories.to_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/Tuning/BestModels/ablations_finallGRAPH.csv')

In [26]:
df_histories

,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates
0,118.709888,0.998855,0.951145,1.870439,1.885224,1.000000e-05
1,116.361754,0.996947,0.948092,1.845734,1.920276,1.000000e-06
2,120.137696,1.000000,0.952672,1.897879,1.873279,1.000000e-07
3,73.079383,0.993130,0.951145,1.922092,1.906677,1.000000e-05
4,68.535003,0.987786,0.935878,1.956521,2.046659,1.000000e-06
5,69.457429,0.990076,0.945038,1.960772,2.067834,1.000000e-07


In [27]:
from sklearn.metrics import matthews_corrcoef, precision_recall_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np

def calculate_mcc_multiclass(y_true, y_pred_probs):
    y_pred_labels = np.argmax(y_pred_probs, axis=1)
    # Convertir etiquetas verdaderas one-hot a etiquetas enteras si es necesario
    #y_true_labels = np.argmax(y_true, axis=1) if y_true.ndim > 1 else y_true
    return matthews_corrcoef(y_true, y_pred_labels)

def calculate_auc_pr_multiclass(y_true, y_pred_probs, average='macro'):
    n_classes = y_pred_probs.shape[1]
    #y_true_labels = np.argmax(y_true, axis=1) if y_true.ndim > 1 else y_true
    y_true_bin = label_binarize(y_true, classes=range(n_classes))

    auc_pr_list = []
    for i in range(n_classes):
        precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_pred_probs[:, i])
        auc_pr_list.append(auc(recall, precision))

    if average == 'macro':
        return np.mean(auc_pr_list)
    elif average == 'weighted':
        class_counts = y_true_bin.sum(axis=0)
        return np.average(auc_pr_list, weights=class_counts)
    else:
        return auc_pr_list


In [28]:
from tensorflow import keras
accuracies_predict=[]
loss_predict=[]
times_predict=[]
mccs_predict=[]
aucpr_predict=[]

Model_Xtest=[Xp_test,Xcode_test,Xs_test,Xt_test,Xf_test]
for i,my_model in enumerate(models):
  model=keras.models.load_model('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/Tuning/BestModels/ablationsGRAPH_finall_{0}.keras'.format(models_names[i]))
  if(i==1):
    Model_Xtest=[Xp_test,Xs_test,Xt_test,Xf_test]
  if(i==2):
    Model_Xtest=[Xcode_test,Xs_test,Xt_test,Xf_test]
  if(i==3):
    Model_Xtest=[Xs_test,Xt_test,Xf_test]
  if(i==4):
    Model_Xtest=[Xp_test,Xcode_test]
  if(i==5):
    Model_Xtest=[Xcode_test]

  evaluate=model.evaluate(Model_Xtest,y_test)

  t1=time()
  y_pred=model.predict(Model_Xtest)
  t2=time()
  mcc=calculate_mcc_multiclass(y_test, y_pred)
  auc_pr=calculate_auc_pr_multiclass(y_test, y_pred)

  times_predict.append(t2-t1)
  accuracies_predict.append(evaluate[1])
  loss_predict.append(evaluate[0])
  mccs_predict.append(mcc)
  aucpr_predict.append(auc_pr)


  accuracy=evaluate[1]
  loss=evaluate[0]

  print("accuracy",accuracy)
  print("loss",loss)
  print("mcc",mcc)
  print("auc_pr",auc_pr)
  print("time predict",t2-t1)

26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - accuracy: 0.9369 - loss: 0.2687
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


accuracy 0.9279609322547913
loss 0.3001468777656555
mcc 0.9032261016991451
auc_pr 0.8698093018212851
time predict 2.617314100265503
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 42ms/step - accuracy: 0.9190 - loss: 0.3081
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step
accuracy 0.9133089184761047
loss 0.32230594754219055
mcc 0.8840633702601819
auc_pr 0.8664312453051128
time predict 1.6611194610595703


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - accuracy: 0.9348 - loss: 0.2632
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step
accuracy 0.9279609322547913
loss 0.2872338593006134
mcc 0.9035599214377195
auc_pr 0.875401926171749
time predict 1.6257836818695068


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9327 - loss: 0.2545
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step
accuracy 0.9242979288101196
loss 0.3061523735523224
mcc 0.8983441040388805
auc_pr 0.8700568119278562
time predict 1.1049795150756836


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.9288 - loss: 0.2793
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step
accuracy 0.9230769276618958
loss 0.3254510760307312
mcc 0.8968187913214917
auc_pr 0.8733754435495793
time predict 0.8345656394958496


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


In [29]:
df_histories['accuracies_predict']=accuracies_predict
df_histories['loss_predict']=loss_predict
df_histories['times_predict']=times_predict
df_histories['mccs']=mccs_predict
df_histories['aucpr']=aucpr_predict
df_histories.to_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/Tuning/BestModels/early_best_finall_predictGRAPH.csv')

In [30]:
df_histories['model']=models_names

In [31]:
df_histories

,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,model
0,118.709888,0.998855,0.951145,1.870439,1.885224,1.000000e-05,0.927961,0.300147,2.617314,0.903226,0.869809,problemall
1,116.361754,0.996947,0.948092,1.845734,1.920276,1.000000e-06,0.913309,0.322306,1.661119,0.884063,0.866431,problem_gries
2,120.137696,1.000000,0.952672,1.897879,1.873279,1.000000e-07,0.927961,0.287234,1.625784,0.903560,0.875402,code_gries
3,73.079383,0.993130,0.951145,1.922092,1.906677,1.000000e-05,0.915751,0.319601,1.329534,0.887130,0.863842,gries
4,68.535003,0.987786,0.935878,1.956521,2.046659,1.000000e-06,0.924298,0.306152,1.104980,0.898344,0.870057,problem_code
5,69.457429,0.990076,0.945038,1.960772,2.067834,1.000000e-07,0.923077,0.325451,0.834566,0.896819,0.873375,code


In [34]:
df_histories[(df_histories['aucpr']>0.87)]

,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,model
2,120.137696,1.000000,0.952672,1.897879,1.873279,1.000000e-07,0.927961,0.287234,1.625784,0.903560,0.875402,code_gries
4,68.535003,0.987786,0.935878,1.956521,2.046659,1.000000e-06,0.924298,0.306152,1.104980,0.898344,0.870057,problem_code
5,69.457429,0.990076,0.945038,1.960772,2.067834,1.000000e-07,0.923077,0.325451,0.834566,0.896819,0.873375,code


In [35]:
df_histories[(df_histories['aucpr']>0.87)&(df_histories['max_accuracies']-df_histories['max_val_accuracies']<0.06)]

,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,model
2,120.137696,1.000000,0.952672,1.897879,1.873279,1.000000e-07,0.927961,0.287234,1.625784,0.903560,0.875402,code_gries
4,68.535003,0.987786,0.935878,1.956521,2.046659,1.000000e-06,0.924298,0.306152,1.104980,0.898344,0.870057,problem_code
5,69.457429,0.990076,0.945038,1.960772,2.067834,1.000000e-07,0.923077,0.325451,0.834566,0.896819,0.873375,code


Early stoping 40 the best